# Notebook 1: Black-Scholes Fundamentals

This notebook works through the Black-Scholes model from the closed-form formula down
to numerical verification with the GPU Monte Carlo engine. Each section makes a concrete
prediction — a price, a convergence rate, a parity relation — and then checks it against
a live simulation. It is a good place to build intuition for where theory and computation
agree, and by how much.

In [ ]:
import sys, os
# Works whether you launch Jupyter from the repo root or from notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'build'))

import mc_engine
import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless — remove if running interactively
import matplotlib.pyplot as plt
from scipy.stats import norm
import math, time

print('mc_engine loaded:', mc_engine.__file__)

## Section 1 — The Black-Scholes Formula

Under the Black-Scholes model the stock price follows geometric Brownian motion:

$$dS = rS\,dt + \sigma S\,dW$$

The **risk-neutral** closed-form price of a European **call** is:

$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$

and of a European **put**:

$$P = K e^{-rT} N(-d_2) - S_0 N(-d_1)$$

where

$$d_1 = \frac{\ln(S_0/K) + (r + \tfrac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

| Symbol | Meaning |
|--------|---------|
| $S_0$  | Current spot price |
| $K$    | Strike price |
| $r$    | Continuously compounded risk-free rate |
| $\sigma$ | Constant volatility |
| $T$    | Time to expiry (years) |
| $N(\cdot)$ | Standard normal CDF |

In [ ]:
def bs_call(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def bs_put(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bs_greeks(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * np.sqrt(T) * norm.pdf(d1)
    theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
             - r * K * np.exp(-r * T) * norm.cdf(d2))
    rho   = K * T * np.exp(-r * T) * norm.cdf(d2)
    return dict(delta=delta, gamma=gamma, vega=vega, theta=theta, rho=rho)

# Textbook example
S, K, r, sigma, T = 100.0, 105.0, 0.05, 0.20, 1.0

call_price = bs_call(S, K, r, sigma, T)
put_price  = bs_put(S, K, r, sigma, T)

print(f'Black-Scholes call price: {call_price:.4f}  (expect ≈ 8.02)')
print(f'Black-Scholes put  price: {put_price:.4f}')

## Section 2 — Monte Carlo Verification

Monte Carlo prices a European option by simulating many paths under the risk-neutral
measure and averaging the discounted payoffs:

$$\hat{C} = e^{-rT} \frac{1}{N} \sum_{i=1}^{N} \max(S_T^{(i)} - K,\, 0)$$

The estimator is unbiased and its standard error decays as $O(1/\sqrt{N})$.

In [ ]:
# Single pricing call at 1M paths
pricer = mc_engine.GBMPricer(
    spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
    n_paths=1_000_000, n_steps=252, use_gpu=True)
result = pricer.price_european_call()

analytical = bs_call(S, K, r, sigma, T)
err        = abs(result.price - analytical)
n_sigmas   = err / result.standard_error

print(f'MC price      : {result.price:.4f}')
print(f'Analytical    : {analytical:.4f}')
print(f'Standard error: {result.standard_error:.4f}')
print(f'|error| / SE  : {n_sigmas:.2f}σ  (expect < 3)')

In [ ]:
# Convergence study
path_counts = [10_000, 100_000, 1_000_000, 10_000_000]
errors = []

for n in path_counts:
    p = mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=n, n_steps=252, use_gpu=True)
    res = p.price_european_call()
    err = abs(res.price - analytical)
    errors.append(err)
    print(f'  N={n:>10,}: price={res.price:.4f}, |err|={err:.4f}, SE={res.standard_error:.4f}')

# Plot
ns = np.array(path_counts, dtype=float)
C  = errors[0] * np.sqrt(path_counts[0])   # scale factor for O(1/sqrt(N)) line

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ns, errors, 'bo-', linewidth=2, markersize=6, label='|MC error|')
ax.loglog(ns, C / np.sqrt(ns), 'r--', linewidth=1.5, label=r'$C/\sqrt{N}$  (O(1/√N))')
ax.set_xlabel('Number of Paths', fontsize=13)
ax.set_ylabel('Absolute Pricing Error', fontsize=13)
ax.set_title('Monte Carlo Convergence: European Call vs Black-Scholes', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()
print('Error decreases proportionally to 1/√N — confirming MC convergence rate.')

## Section 3 — Put-Call Parity

For European options on a non-dividend-paying stock, put-call parity states:

$$C - P = S_0 - K e^{-rT}$$

This is a **model-free** no-arbitrage result — it must hold regardless of the dynamics
assumed, including in Monte Carlo simulations that share the same random seed.  
We verify it for five different strikes.

In [ ]:
strikes = [80.0, 90.0, 100.0, 110.0, 120.0]

hdr = f"{'Strike':>7} | {'Call':>8} | {'Put':>8} | {'C-P':>8} | {'S-Ke^-rT':>10} | {'Diff':>8}"
sep = '-' * len(hdr)
print(sep)
print(hdr)
print(sep)

for Ki in strikes:
    p = mc_engine.GBMPricer(
        spot=S, strike=Ki, rate=r, volatility=sigma, maturity=T,
        n_paths=1_000_000, n_steps=252, use_gpu=True)
    rc = p.price_european_call()
    rp = p.price_european_put()
    parity = S - Ki * math.exp(-r * T)
    diff   = abs(rc.price - rp.price - parity)
    tol    = 3.0 * (rc.standard_error + rp.standard_error)
    flag   = 'OK' if diff < tol else 'FAIL'
    print(f"{Ki:>7.0f} | {rc.price:>8.3f} | {rp.price:>8.3f} | "
          f"{rc.price-rp.price:>8.3f} | {parity:>10.3f} | {diff:>8.4f} [{flag}]")
print(sep)

## Section 4 — Greeks: Analytical vs Monte Carlo

Greeks measure the sensitivity of the option price to each input parameter.
The Black-Scholes closed-form Greeks for a **call** are:

| Greek | Formula | Intuition |
|-------|---------|----------|
| **Delta** $\Delta = \partial C/\partial S$ | $N(d_1)$ | Shares needed to hedge |
| **Gamma** $\Gamma = \partial^2 C/\partial S^2$ | $N'(d_1)/(S\sigma\sqrt{T})$ | Rate of change of Delta |
| **Vega** $\mathcal{V} = \partial C/\partial \sigma$ | $S\sqrt{T}N'(d_1)$ | Sensitivity to vol |
| **Theta** $\Theta = \partial C/\partial t$ | $-(S N'(d_1)\sigma)/(2\sqrt{T}) - rKe^{-rT}N(d_2)$ | Time decay |
| **Rho** $\rho = \partial C/\partial r$ | $KTe^{-rT}N(d_2)$ | Interest-rate sensitivity |

The MC engine computes Greeks by finite differences with bumped paths (central
difference where possible) sharing random numbers via the common random numbers (CRN)
technique to reduce variance.

In [ ]:
# ATM option for clearest comparison
Katm = 100.0

pricer_atm = mc_engine.GBMPricer(
    spot=S, strike=Katm, rate=r, volatility=sigma, maturity=T,
    n_paths=2_000_000, n_steps=252, use_gpu=True)

mc_g  = pricer_atm.compute_greeks()
bs_g  = bs_greeks(S, Katm, r, sigma, T)

hdr = f"{'Greek':<6} | {'Analytical':>12} | {'Monte Carlo':>12} | {'Rel Error':>10}"
sep = '-' * len(hdr)
print(sep)
print(hdr)
print(sep)

greek_pairs = [
    ('delta', mc_g.delta,  bs_g['delta']),
    ('gamma', mc_g.gamma,  bs_g['gamma']),
    ('vega',  mc_g.vega,   bs_g['vega']),
    ('theta', mc_g.theta,  bs_g['theta']),
    ('rho',   mc_g.rho,    bs_g['rho']),
]
for name, mc_val, bs_val in greek_pairs:
    rel = abs(mc_val - bs_val) / abs(bs_val) if abs(bs_val) > 1e-10 else abs(mc_val - bs_val)
    print(f"{name:<6} | {bs_val:>12.4f} | {mc_val:>12.4f} | {rel:>9.2%}")
print(sep)

## Section 5 — Delta as a Function of Spot

Delta $\Delta = N(d_1)$ is bounded between 0 and 1 for a call option.  
It traces an S-curve in spot price:

- Deep **in-the-money** ($S \gg K$): $\Delta \to 1$ — option behaves like the stock
- **At-the-money** ($S \approx K$): $\Delta \approx 0.5$
- Deep **out-of-the-money** ($S \ll K$): $\Delta \to 0$ — option expires worthless

Delta-hedging: holding $-\Delta$ shares per option creates a locally risk-free position.
As spot moves, $\Delta$ changes (captured by Gamma), requiring continuous rebalancing.

In [ ]:
# Analytical delta curve
spots_dense = np.linspace(60, 140, 200)
delta_analytical = np.array([bs_greeks(s, K, r, sigma, T)['delta'] for s in spots_dense])

# MC delta at a handful of spot values (each is one pricing call)
spots_mc    = [65, 80, 90, 100, 110, 120, 135]
delta_mc    = []
for s in spots_mc:
    p = mc_engine.GBMPricer(
        spot=float(s), strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=1_000_000, n_steps=252, use_gpu=True)
    delta_mc.append(p.compute_greeks().delta)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(spots_dense, delta_analytical, 'b-', linewidth=2, label='Analytical Delta')
ax.scatter(spots_mc, delta_mc, color='red', zorder=5, s=60, label='Monte Carlo Delta')
ax.axvline(K, color='gray', linestyle='--', linewidth=1, label=f'Strike K={K}')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
ax.set_xlabel('Spot Price  S', fontsize=13)
ax.set_ylabel('Delta  Δ', fontsize=13)
ax.set_title('Delta vs Spot: European Call (K=105, σ=0.20, T=1yr)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()